In [ ]:
!pip install -q ase networkx scipy matplotlib pandas

import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from ase.io import read
from scipy.spatial.distance import pdist, squareform
import networkx as nx
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("step1_output")
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# Download once; -nc / -n skip re-downloading and re-extracting on reruns
!wget -q -nc "https://zenodo.org/records/14184621/files/CoREMOF2019_public_v2_20241119.zip?download=1" -O core_mof_2019.zip
!unzip -qn core_mof_2019.zip -d core_mof_2019
import glob
all_cifs = glob.glob("core_mof_2019/**/CR/**/*.cif", recursive=True)
print(len(all_cifs), "CR (computation-ready) CIFs available")


In [ ]:
# ============================================================
# Coarse-graining: CIF -> block graph -> influential subgraph
# (identical logic to your original Cell 1, wrapped in a function)
# ============================================================

def bond_cutoff(el1, el2):
    metals = {"Zn", "Cu", "Ni", "Co", "Fe", "Al", "Cr", "Mn", "Zr", "Ti"}
    pair = {el1, el2}
    if el1 in metals or el2 in metals:
        return 2.45
    if pair <= {"C", "O", "N"}:
        return 1.70
    if "H" in pair or "F" in pair:
        return 1.45
    return 1.90

METALS = {"Zn", "Cu", "Ni", "Co", "Fe", "Al", "Cr", "Mn", "Zr", "Ti"}


def build_block_graph(cif_path, top_percent=0.30, verbose=True):
    mof = read(cif_path)
    pos = mof.get_positions().copy()
    symbols = np.array(mof.get_chemical_symbols())
    n_atoms = len(mof)

    if verbose:
        print(f"[{cif_path}] Loaded: {n_atoms} atoms | Formula: {mof.get_chemical_formula()}")

    dist = squareform(pdist(pos))

    adj = np.zeros((n_atoms, n_atoms), dtype=bool)
    for i in range(n_atoms):
        for j in range(i+1, n_atoms):
            if 0.4 < dist[i, j] < bond_cutoff(symbols[i], symbols[j]):
                adj[i, j] = adj[j, i] = True

    metal_idx = [i for i, s in enumerate(symbols) if s in METALS]
    if not metal_idx:
        raise ValueError(f"No recognized metal atoms found in {cif_path} (METALS={METALS})")

    node_atoms = set(metal_idx)
    for m in metal_idx:
        for o in range(n_atoms):
            if symbols[o] == "O" and adj[m, o]:
                node_atoms.add(o)
    node_atoms = sorted(node_atoms)

    organic = [i for i in range(n_atoms) if i not in node_atoms]
    G_org = nx.Graph()
    G_org.add_nodes_from(organic)
    for i in organic:
        for j in organic:
            if i < j and adj[i, j]:
                G_org.add_edge(i, j)

    linker_components = [sorted(c) for c in nx.connected_components(G_org)]

    building_blocks = [node_atoms] + linker_components
    bb_types = ["metal"] + ["linker"] * len(linker_components)
    K = len(building_blocks)

    atom_to_bb = {}
    for bb_id, atoms in enumerate(building_blocks):
        for a in atoms:
            atom_to_bb[a] = bb_id

    block_edges = Counter()
    for i in range(n_atoms):
        for j in range(i+1, n_atoms):
            if adj[i, j]:
                bi, bj = atom_to_bb[i], atom_to_bb[j]
                if bi != bj:
                    edge = tuple(sorted((bi, bj)))
                    block_edges[edge] += 1

    G_block = nx.Graph()
    G_block.add_nodes_from(range(K))
    for (bi, bj), w in block_edges.items():
        G_block.add_edge(bi, bj, weight=w)

    degrees = dict(G_block.degree())
    sorted_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)
    n_influential = max(1, int(np.ceil(K * top_percent)))
    influential_nodes = [node for node, deg in sorted_nodes[:n_influential]]

    G_inf = G_block.subgraph(influential_nodes).copy()

    if verbose:
        print(f"[{cif_path}] K={K} blocks, {G_block.number_of_edges()} edges, "
              f"{n_influential} influential nodes")

    return {
        "G_block": G_block,
        "G_inf": G_inf,
        "influential_nodes": influential_nodes,
        "bb_types": bb_types,
        "K": K,
    }


# ============================================================
# iNMFA spectrum (identical logic to your Cell 2, wrapped)
# ============================================================

def box_covering(G, rb, dist=None, seed=None):
    if dist is None:
        dist = dict(nx.all_pairs_shortest_path_length(G))

    rng = random.Random(seed)
    uncovered = set(G.nodes())
    node_box = {}
    box_size = {}
    box_id = 0

    nodes_order = list(G.nodes())
    rng.shuffle(nodes_order)

    for center in nodes_order:
        if center not in uncovered:
            continue
        members = [n for n in uncovered
                   if n in dist[center] and dist[center][n] <= rb]
        if not members:
            members = [center]
        for m in members:
            node_box[m] = box_id
            uncovered.discard(m)
        box_size[box_id] = len(members)
        box_id += 1
        if not uncovered:
            break

    return node_box, box_size


def probability_measures(G, influential_nodes, radii, n_trials=5, seed=0):
    N = G.number_of_nodes()
    dist = dict(nx.all_pairs_shortest_path_length(G))
    pr = {r: {node: [] for node in influential_nodes} for r in radii}

    for trial in range(n_trials):
        for r in radii:
            node_box, box_size = box_covering(G, r, dist=dist, seed=seed + trial)
            for node in influential_nodes:
                if node in node_box:
                    pr[r][node].append(box_size[node_box[node]] / N)

    pr_avg = {r: {node: np.mean(vals) for node, vals in d.items() if vals}
              for r, d in pr.items()}
    return pr_avg


def partition_function(pr_avg, q, radii):
    Pq = []
    for r in radii:
        vals = np.array(list(pr_avg[r].values()))
        vals = vals[vals > 0]
        Pq.append(np.sum(vals ** q))
    return np.array(Pq)


def compute_tau(pr_avg, radii, r_N, q_values):
    tau = []
    x = np.log(np.array(radii) / r_N)
    for q in q_values:
        Pq = partition_function(pr_avg, q, radii)
        y = np.log(Pq)
        mask = np.isfinite(y)
        if mask.sum() < 2:
            tau.append(np.nan)
            continue
        slope, _ = np.polyfit(x[mask], y[mask], 1)
        tau.append(slope)
    return np.array(tau)


def legendre_transform(q_values, tau):
    alpha = np.gradient(tau, q_values)
    f_alpha = q_values * alpha - tau
    return alpha, f_alpha


def asymmetry_metric(alpha, q_values):
    idx0 = np.argmin(np.abs(q_values - 0))
    alpha0 = alpha[idx0]
    alpha_min, alpha_max = np.nanmin(alpha), np.nanmax(alpha)
    return np.log((alpha0 - alpha_min) / (alpha_max - alpha0))


def run_inmfa(G, influential_nodes, q_values=np.linspace(-10, 10, 41),
              n_box_trials=5, seed=0):
    if nx.is_connected(G):
        diameter = nx.diameter(G)
    else:
        comp_diams = [nx.diameter(G.subgraph(c)) for c in nx.connected_components(G)
                      if len(c) > 1]
        diameter = max(comp_diams) if comp_diams else 0

    diameter = max(diameter, 1)  # safeguard: avoid empty radii on tiny subgraphs
    radii = list(range(1, diameter + 1))
    r_N = diameter

    pr_avg = probability_measures(G, influential_nodes, radii,
                                   n_trials=n_box_trials, seed=seed)
    tau = compute_tau(pr_avg, radii, r_N, q_values)
    alpha, f_alpha = legendre_transform(q_values, tau)
    A = asymmetry_metric(alpha, q_values)

    return {
        "radii": radii,
        "q_values": q_values,
        "tau": tau,
        "alpha": alpha,
        "f_alpha": f_alpha,
        "asymmetry": A,
    }


# ============================================================
# Full per-MOF pipeline: CIF -> multifractal spectrum
# ============================================================

def analyze_mof(cif_path, top_percent=0.30, q_values=np.linspace(-10, 10, 41),
                n_box_trials=5, seed=0, verbose=True):
    block_data = build_block_graph(cif_path, top_percent=top_percent, verbose=verbose)
    spectrum = run_inmfa(block_data["G_inf"], block_data["influential_nodes"],
                          q_values=q_values, n_box_trials=n_box_trials, seed=seed)
    result = {**block_data, **spectrum, "cif_path": cif_path}
    result["n_influential"] = len(block_data["influential_nodes"])
    return result

In [ ]:
# ============================================================
# Pick a REFERENCE set that's structurally similar, not random.
# We do this by grouping MOFs by topology (the underlying net
# shape) and taking many MOFs that all share the SAME topology.
# Similar topology -> similar block-graph -> similar spectrum
# -> an actual band instead of a scattered mess.
# ============================================================

import os

topo_path = glob.glob("core_mof_2019/**/topologies_individual_with_domains.tsv", recursive=True)[0]
topo = pd.read_csv(topo_path, sep="\t")
print("Columns found in topology file:", topo.columns.tolist())
print(topo.head())

# Auto-detect which column holds the MOF name/refcode and which holds the topology label.
# (Prints above let you sanity-check these guesses; adjust name_col/topo_col manually if wrong.)
name_col = next((c for c in topo.columns if any(k in c.lower() for k in ("name", "refcode", "filename", "id"))), topo.columns[0])
topo_col = next((c for c in topo.columns if any(k in c.lower() for k in ("topo", "net"))), topo.columns[1])
print(f"\nUsing name column: {name_col!r}   topology column: {topo_col!r}")

# Pick the single most common topology in the database -> guarantees a large, similar pool
top_topology = topo[topo_col].value_counts().idxmax()
n_matches = (topo[topo_col] == top_topology).sum()
print(f"Most common topology: {top_topology!r}  ({n_matches} MOFs share it)")

matching_names = set(topo.loc[topo[topo_col] == top_topology, name_col].astype(str))

# Map refcode (e.g. 'ABAVIJ') -> full CIF path on disk
def refcode_from_path(p):
    return os.path.basename(p).split("_")[0]

cif_by_refcode = {refcode_from_path(p): p for p in all_cifs}

N_REFERENCE = 15
N_HOLDOUT = 3  # extra MOFs from the SAME topology, held out as "should be inside the band" test candidates

matched_cifs = [cif_by_refcode[n] for n in matching_names if n in cif_by_refcode]
print(f"{len(matched_cifs)} of those also have a downloaded CR CIF file")

REFERENCE_CIFS = matched_cifs[:N_REFERENCE]
HOLDOUT_CIFS = matched_cifs[N_REFERENCE:N_REFERENCE + N_HOLDOUT]

print(f"\nReference set: {len(REFERENCE_CIFS)} MOFs (same topology: {top_topology})")
print(f"Holdout set (candidates to test against the band): {len(HOLDOUT_CIFS)} MOFs")


In [ ]:
reference_results = []
skipped = []

for cif in REFERENCE_CIFS:
    try:
        res = analyze_mof(cif)
        reference_results.append(res)
        print(f"✓ {cif}: K={res['K']}, n_influential={res['n_influential']}, A={res['asymmetry']:.3f}")
    except Exception as e:
        print(f"✗ Skipping {cif}: {e}")
        skipped.append(cif)

if len(reference_results) < 2:
    raise RuntimeError("Need at least 2 successfully processed reference MOFs to build a band.")

def clean_curve(alpha, f_alpha):
    """Drop NaNs, sort by alpha, and average duplicate alpha values."""
    alpha = np.asarray(alpha)
    f_alpha = np.asarray(f_alpha)
    mask = np.isfinite(alpha) & np.isfinite(f_alpha)
    a, f = alpha[mask], f_alpha[mask]
    order = np.argsort(a)
    a, f = a[order], f[order]
    a_unique, inverse = np.unique(a, return_inverse=True)
    f_unique = np.array([f[inverse == k].mean() for k in range(len(a_unique))])
    return a_unique, f_unique

# Build a common alpha grid spanning all reference MOFs
all_alphas = np.concatenate([clean_curve(r["alpha"], r["f_alpha"])[0] for r in reference_results])
grid_min, grid_max = np.nanmin(all_alphas), np.nanmax(all_alphas)
alpha_grid = np.linspace(grid_min, grid_max, 200)

# Interpolate each reference MOF's curve onto that grid
# (left/right = NaN so we never extrapolate beyond a MOF's own alpha range)
interp_curves = []
kept_labels = []
for r in reference_results:
    a, f = clean_curve(r["alpha"], r["f_alpha"])
    if len(a) < 2:
        print(f"⚠ {r['cif_path']}: too few valid spectrum points, excluded from band")
        continue
    fi = np.interp(alpha_grid, a, f, left=np.nan, right=np.nan)
    interp_curves.append(fi)
    kept_labels.append(r["cif_path"])

interp_curves = np.array(interp_curves)

# Min-max envelope band
lower = np.nanmin(interp_curves, axis=0)
upper = np.nanmax(interp_curves, axis=0)
valid = np.isfinite(lower) & np.isfinite(upper)

plt.figure(figsize=(7, 6))
plt.fill_between(alpha_grid[valid], lower[valid], upper[valid],
                  color="steelblue", alpha=0.3, label="Reference band (min-max envelope)")
for label, fi in zip(kept_labels, interp_curves):
    plt.plot(alpha_grid, fi, color="steelblue", alpha=0.5, lw=1, label=label)
plt.xlabel(r"$\alpha(q)$")
plt.ylabel(r"$f(\alpha)$")
plt.title(f"Multifractal spectrum band (topology: {top_topology})")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "xe_mof_band.png", dpi=150)
plt.show()


In [ ]:
# ------------------------------------------------------------
# Test a candidate MOF against the band.
# Default: use a held-out MOF from the SAME topology (should score high).
# Swap in your own generated/candidate CIF path to test something else.
# ------------------------------------------------------------
CANDIDATE_CIF = HOLDOUT_CIFS[0] if HOLDOUT_CIFS else all_cifs[0]

candidate_result = analyze_mof(CANDIDATE_CIF)
a_c, f_c = clean_curve(candidate_result["alpha"], candidate_result["f_alpha"])
f_c_interp = np.interp(alpha_grid, a_c, f_c, left=np.nan, right=np.nan)

compare_mask = np.isfinite(f_c_interp) & np.isfinite(lower) & np.isfinite(upper)
n_valid = compare_mask.sum()

if n_valid == 0:
    print("⚠ Candidate MOF's alpha-range does not overlap the reference band at all — cannot compare.")
    pct_inside = None
else:
    inside = (f_c_interp[compare_mask] >= lower[compare_mask]) & (f_c_interp[compare_mask] <= upper[compare_mask])
    pct_inside = 100 * inside.sum() / n_valid
    print(f"Candidate MOF: {pct_inside:.1f}% of comparable spectrum points "
          f"fall inside the reference band ({n_valid} points compared).")

plt.figure(figsize=(7, 6))
plt.fill_between(alpha_grid[valid], lower[valid], upper[valid],
                  color="steelblue", alpha=0.3, label="Reference band")
plt.plot(alpha_grid, f_c_interp, color="crimson", lw=2, label="Candidate MOF")
title = "Candidate MOF vs reference band"
if pct_inside is not None:
    title += f" ({pct_inside:.1f}% inside)"
plt.title(title)
plt.xlabel(r"$\alpha(q)$")
plt.ylabel(r"$f(\alpha)$")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "candidate_vs_band.png", dpi=150)
plt.show()
